# Lesson 7: Observability & Debugging

## What You'll Learn

1. **Why observability matters** — Agents are opaque by default
2. **Structured tracing** — Track every step of an agent run
3. **Cost tracking** — Know exactly how much each query costs
4. **Performance analysis** — P50/P95 latency, token usage trends
5. **Error analysis** — Categorize and debug failures
6. **PydanticAI message inspection** — Reading raw tool call traces
7. **Production integrations** — Logfire and Langfuse overview

---

## The Observability Problem

Agents are **non-deterministic black boxes**. Without tracing:
- You can't tell WHY an answer is wrong (bad SQL? wrong tool? hallucination?)
- You can't optimize cost (which queries burn the most tokens?)
- You can't debug intermittent failures
- You have no performance baselines

### Three Pillars of Agent Observability

| Pillar | What | Tools |
|--------|------|-------|
| **Traces** | Step-by-step execution record | Custom tracer, Logfire, Langfuse |
| **Metrics** | Aggregated numbers (latency, cost, success rate) | Dashboards, alerting |
| **Logs** | Detailed debug output | Structured logging |

---

## Setup

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import time
import json
import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic_ai import Agent, RunContext, ModelRetry

from analyst.observability.tracing import Span, Trace, AgentTracer

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

Loaded: sales ((40, 8)), employees ((20, 8))


In [2]:
from pydantic_ai.exceptions import ModelHTTPError


def run_agent_checked(agent, *args, **kwargs):
    """Run agent safely and surface LM Studio template failures clearly."""
    try:
        return agent.run_sync(*args, **kwargs)
    except ModelHTTPError as exc:
        message = str(exc)
        if "No user query found in messages" in message:
            raise RuntimeError(
                "LM Studio prompt template is incompatible with this notebook flow.\n"
                "Fix in LM Studio:\n"
                "1) Use an lmstudio-community model/template for your model family.\n"
                "2) In My Models -> Prompt Template, remove logic that raises when no user query is found.\n"
                "3) Disable model thinking/reasoning output for API calls with strict structured outputs.\n"
                "4) Reload the model and rerun from the setup cell."
            ) from exc
        raise


---

## Part 1: Manual Tracing — Understanding the Building Blocks

Before using any framework, let's build tracing from scratch to understand
exactly what happens during an agent run.

### Core concepts: Spans and Traces

```
Trace (one complete agent run)
│
├── Span: "llm_decision"      ← LLM decides which tool to call
│     type: llm_call
│     duration: 120ms
│     tokens: 150
│
├── Span: "run_sql"            ← Tool executes
│     type: tool_call
│     duration: 8ms
│     input: "SELECT SUM(revenue)..."
│     output: "45230.50"
│
└── Span: "synthesize"         ← LLM formats final answer
      type: llm_call
      duration: 80ms
      tokens: 100
```

**Span** = one operation (an LLM call, a tool call, code execution).
Each span tracks: name, type, timing, input/output, token count, errors.

**Trace** = a collection of spans for one complete user query.
Each trace tracks: all spans, total tokens, total cost, success/failure.

### Why build it ourselves first?

Frameworks like Logfire auto-trace everything, but you won't understand
WHAT they're tracing or WHY unless you've built it manually. This lesson
builds the intuition, then shows how frameworks automate it.

In [3]:
# -----------------------------------------------------------------
# Spans and Traces — the core primitives
# -----------------------------------------------------------------

# A Span represents one unit of work
span = Span(name="run_sql", span_type="tool_call", input_data="SELECT SUM(revenue) FROM sales")
time.sleep(0.1)  # Simulate work
span.finish(output="45230.50")

print("Single span:")
print(json.dumps(span.to_dict(), indent=2))

# A Trace groups related spans
trace = Trace(trace_id="demo_001", question="What is the total revenue?", model="gpt-4o-mini")

# Span 1: LLM decides to call a tool
s1 = Span(name="llm_decision", span_type="llm_call", input_data="What is the total revenue?")
time.sleep(0.05)
s1.finish(output="I should run a SQL query to sum the revenue column.")
s1.tokens_used = 150
trace.add_span(s1)

# Span 2: Tool execution
s2 = Span(name="run_sql", span_type="tool_call", input_data="SELECT SUM(revenue) FROM sales")
time.sleep(0.02)
s2.finish(output="45230.50")
trace.add_span(s2)

# Span 3: LLM synthesizes final answer
s3 = Span(name="synthesize", span_type="llm_call", input_data="SQL result: 45230.50")
time.sleep(0.03)
s3.finish(output="The total revenue is $45,230.50.")
s3.tokens_used = 100
trace.add_span(s3)

trace.finish(answer="The total revenue is $45,230.50.")

print(f"\n{trace.summary()}")
print(f"\nTimeline:")
for s in trace.spans:
    print(f"  [{s.span_type:>10}] {s.name:<20} {s.duration_ms:>6.1f}ms  tokens={s.tokens_used}")

Single span:
{
  "name": "run_sql",
  "type": "tool_call",
  "duration_ms": 105.2,
  "tokens": 0,
  "error": null,
  "input": "SELECT SUM(revenue) FROM sales",
  "output": "45230.50",
  "metadata": {}
}

Trace: demo_001
  Question: What is the total revenue?
  Model: gpt-4o-mini
  Duration: 112ms
  Tokens: 250
  Cost: $0.0000
  Tool calls: 1
  LLM calls: 2
  Errors: 0
  Success: True

Timeline:
  [  llm_call] llm_decision           55.1ms  tokens=150
  [ tool_call] run_sql                21.3ms  tokens=0
  [  llm_call] synthesize             35.1ms  tokens=100


---

## Part 2: Traced Agent — Automatic Instrumentation

Now let's wire the tracer into a real agent so every tool call and
LLM interaction is captured automatically.

### How auto-tracing works

The key insight: we instrument at the **tool level**, not the LLM level.
PydanticAI handles the LLM calls internally, but our `@agent.tool` functions
are where we add tracing:

```
User question
     │
     ▼
┌─ traced_query() ──────────────────────────────┐
│  1. Create a Trace                             │
│  2. Set deps.current_trace = trace             │
│  3. Call agent.run_sync()                      │
│     │                                          │
│     ├─ PydanticAI calls LLM (we can't see)     │
│     ├─ LLM decides to call run_sql             │
│     ├─ run_sql() creates a Span ← OUR CODE     │
│     ├─ Tool executes, span.finish()             │
│     ├─ PydanticAI sends result back to LLM      │
│     └─ LLM produces final answer               │
│                                                │
│  4. Estimate tokens from message history       │
│  5. trace.finish()                             │
└────────────────────────────────────────────────┘
```

### What we CAN vs CAN'T trace natively

| Can Trace | Can't Trace (without Logfire) |
|-----------|-------------------------------|
| Tool call inputs/outputs | Internal LLM prompt construction |
| Tool call timing | LLM inference time (bundled) |
| Total token estimate | Exact token split (input vs output) |
| Errors and retries | LLM reasoning/chain-of-thought |
| Overall latency | Network latency vs compute time |

Logfire (Part 8) fills in the gaps by hooking into PydanticAI internals.

In [4]:
# -----------------------------------------------------------------
# Agent with built-in tracing
# -----------------------------------------------------------------

@dataclass
class TracedDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    tracer: AgentTracer = field(default_factory=AgentTracer)
    current_trace: Trace | None = None


traced_agent = Agent(
    "openai:local-model",
    deps_type=TracedDeps,
    system_prompt=(
        "You are a data analyst. Answer questions using SQL.\n"
        "Always include specific numbers. Be concise."
    ),
    retries=2,
)


@traced_agent.system_prompt
def add_context(ctx: RunContext[TracedDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@traced_agent.tool
def run_sql(ctx: RunContext[TracedDeps], query: str) -> str:
    """Run SQL against available tables using DuckDB."""
    # Start a span for this tool call
    trace = ctx.deps.current_trace
    span = None
    if trace:
        span = ctx.deps.tracer.start_span(trace, "run_sql", "tool_call", query)
    
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(query).fetchdf()
        output = result.to_string()
        if span:
            span.finish(output=output)
        return output
    except Exception as e:
        if span:
            span.finish(error=str(e))
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


def traced_query(deps: TracedDeps, question: str) -> str:
    """Run a query with full tracing."""
    # Start trace
    trace = deps.tracer.start_trace(question, model="openai:local-model")
    deps.current_trace = trace
    
    try:
        result = run_agent_checked(traced_agent, question, deps=deps)
        
        # Estimate tokens from messages
        for msg in result.all_messages():
            msg_str = str(msg)
            token_est = len(msg_str) // 4
            trace.total_tokens += token_est
        
        # Estimate cost
        deps.tracer.estimate_cost(trace)
        trace.finish(answer=result.output, success=True)
        return result.output
    except Exception as e:
        trace.finish(answer="", success=False)
        return f"Error: {e}"


print("Traced agent ready.")

Traced agent ready.


In [5]:
# -----------------------------------------------------------------
# Run several queries with tracing
# -----------------------------------------------------------------

deps = TracedDeps(
    tables={"sales": sales_df, "employees": employees_df},
)

questions = [
    "What is the total revenue in the sales data?",
    "Which region has the highest revenue?",
    "What is the average salary by department?",
    "How many products have revenue above 1000?",
    "What is the correlation between quantity and revenue?",
]

for q in questions:
    print(f"\nQ: {q}")
    answer = traced_query(deps, q)
    print(f"A: {answer[:120]}")

print(f"\nTotal traces recorded: {len(deps.tracer.traces)}")


Q: What is the total revenue in the sales data?
A: 

The total revenue in the sales data is $147,695.10.

Q: Which region has the highest revenue?
A: 

The East region has the highest revenue with a total of $39,311.20.

Q: What is the average salary by department?
A: 

The average salary by department is:
- **Engineering**: $149,400.00
- **Marketing**: $114,500.00
- **Sales**: $110,500

Q: How many products have revenue above 1000?
A: 

6 products have revenue above 1000.

Q: What is the correlation between quantity and revenue?
A: 

The correlation between quantity and revenue is **-0.2157**. This indicates a weak negative relationship, meaning that

Total traces recorded: 5


---

## Part 3: The Performance Dashboard

With traces collected, we can now build a comprehensive view of agent performance.

### What to look for in the dashboard

| Metric | Healthy | Warning | Action |
|--------|---------|---------|--------|
| **Success rate** | > 95% | < 90% | Check error categories |
| **P50 latency** | < 2s | > 5s | Optimize prompts, fewer tools |
| **P95 latency** | < 5s | > 15s | Add timeouts, check outliers |
| **Cost per query** | < $0.01 | > $0.05 | Switch to smaller model for simple queries |
| **Tool calls/query** | 1-2 | > 4 | Agent may be thrashing — improve system prompt |

**P50 vs P95**: P50 is the "typical" experience (half are faster). P95 is
the "worst case" (19 out of 20 are faster). Track both — a fast P50 with
a terrible P95 means some users have a bad experience.

In [6]:
# -----------------------------------------------------------------
# Dashboard: aggregate view of all runs
# -----------------------------------------------------------------

print(deps.tracer.dashboard())

        AGENT PERFORMANCE DASHBOARD
  Total runs:    5
  Success rate:  100%

  Latency:
    P50:  45531ms
    P95:  70746ms
    Mean: 51260ms

  Tokens:
    Mean per run: 0
    Total:        0

  Tool calls:
    Mean per run: 1.0

  Cost:
    Total:    $0.0000
    Per run:  $0.0000


In [7]:
# -----------------------------------------------------------------
# Detailed per-trace analysis
# -----------------------------------------------------------------

print("Per-trace breakdown:\n")
print(f"{'Question':<50} {'Duration':>10} {'Tokens':>8} {'Tools':>6} {'Cost':>10}")
print("-" * 88)

for trace in deps.tracer.traces:
    print(
        f"{trace.question[:48]:<50} "
        f"{trace.duration_ms:>8.0f}ms "
        f"{trace.total_tokens:>8} "
        f"{len(trace.tool_calls):>6} "
        f"${trace.total_cost_usd:>8.4f}"
    )

Per-trace breakdown:

Question                                             Duration   Tokens  Tools       Cost
----------------------------------------------------------------------------------------
What is the total revenue in the sales data?          70746ms        0      1 $  0.0000
Which region has the highest revenue?                 45531ms        0      1 $  0.0000
What is the average salary by department?             43682ms        0      1 $  0.0000
How many products have revenue above 1000?            50943ms        0      1 $  0.0000
What is the correlation between quantity and rev      45397ms        0      1 $  0.0000


In [8]:
# -----------------------------------------------------------------
# Inspect the slowest trace in detail
# -----------------------------------------------------------------

slowest = deps.tracer.get_slowest(1)
if slowest:
    t = slowest[0]
    print(f"SLOWEST TRACE")
    print(f"{'='*50}")
    print(t.summary())
    print(f"\nSpan timeline:")
    for s in t.spans:
        status = "ERR" if s.error else "OK"
        print(f"  [{s.span_type:>10}] {s.name:<20} {s.duration_ms:>7.1f}ms [{status}]")
        if s.input_data:
            print(f"              Input:  {s.input_data[:80]}")
        if s.output_data:
            print(f"              Output: {s.output_data[:80]}")

SLOWEST TRACE
Trace: trace_0001
  Question: What is the total revenue in the sales data?
  Model: openai:local-model
  Duration: 70746ms
  Tokens: 0
  Cost: $0.0000
  Tool calls: 1
  LLM calls: 0
  Errors: 0
  Success: True

Span timeline:
  [ tool_call] run_sql                  8.6ms [OK]
              Input:  SELECT SUM(revenue) AS total_revenue FROM sales
              Output:    total_revenue
0       147695.1


---

## Part 4: PydanticAI Message Inspection

PydanticAI stores the full message history of each run. This is the
most detailed trace you can get — every LLM prompt and response,
every tool call and return.

### Why inspect messages?

When something goes wrong, the message history tells you EXACTLY what happened:
- Did the LLM get the right context?
- Did it call the right tool?
- What SQL did it generate?
- Did the tool return useful data?
- How did the LLM interpret the tool output?

### PydanticAI message types

```
result.all_messages() returns a list of:

ModelRequest     ← Messages sent TO the LLM
  ├── SystemPromptPart    (system prompt text)
  ├── UserPromptPart      (user's question)
  └── ToolReturnPart      (tool results sent back)

ModelResponse    ← Messages FROM the LLM
  ├── TextPart            (text output)
  └── ToolCallPart        (tool call request)
```

Each `ModelRequest` + `ModelResponse` pair is one API round-trip.
The number of pairs tells you how many LLM calls were made.

In [9]:
# -----------------------------------------------------------------
# PydanticAI native message inspection
# -----------------------------------------------------------------

simple_agent = Agent(
    "openai:local-model",
    deps_type=TracedDeps,
    system_prompt="You are a data analyst. Use SQL to answer questions. Be concise.",
    retries=2,
)


@simple_agent.system_prompt
def ctx(ctx: RunContext[TracedDeps]) -> str:
    lines = ["\nTables:"]
    for name, df in ctx.deps.tables.items():
        lines.append(f"  '{name}': {len(df)} rows, cols: {list(df.columns)}")
    return "\n".join(lines)


@simple_agent.tool
def query_sql(ctx: RunContext[TracedDeps], sql: str) -> str:
    """Run SQL query against tables."""
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        return conn.execute(sql).fetchdf().to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


# Run a query and inspect the messages
deps2 = TracedDeps(tables={"sales": sales_df, "employees": employees_df})
result = run_agent_checked(simple_agent, "Which product has the highest total revenue?", deps=deps2)

print(f"Answer: {result.output}\n")
print(f"Total messages in history: {len(result.all_messages())}\n")

# Inspect each message
for i, msg in enumerate(result.all_messages()):
    msg_type = type(msg).__name__
    msg_str = str(msg)[:150]
    print(f"  [{i}] {msg_type}")
    print(f"      {msg_str}")
    print()

Answer: 

The product with the highest total revenue is **Widget A**, with a total revenue of **36,887.7**.

Total messages in history: 4

  [0] ModelRequest
      ModelRequest(parts=[SystemPromptPart(content='You are a data analyst. Use SQL to answer questions. Be concise.', timestamp=datetime.datetime(2026, 3, 

  [1] ModelResponse
      ModelResponse(parts=[ThinkingPart(content='\n\n', id='content', provider_name='openai'), TextPart(content='\n\n'), ToolCallPart(tool_name='query_sql',

  [2] ModelRequest
      ModelRequest(parts=[ToolReturnPart(tool_name='query_sql', content='    product  total_revenue\n0  Widget A        36887.7', tool_call_id='128325422', 

  [3] ModelResponse
      ModelResponse(parts=[ThinkingPart(content='\n\n', id='content', provider_name='openai'), TextPart(content='\n\nThe product with the highest total reve



In [10]:
# -----------------------------------------------------------------
# Parse message types to understand the agent's reasoning flow
# -----------------------------------------------------------------

print("Agent reasoning flow:\n")
step = 0
for msg in result.all_messages():
    msg_type = type(msg).__name__
    
    if "Request" in msg_type and "System" not in msg_type:
        # User or system prompt
        step += 1
        content = str(msg)
        if "UserPrompt" in msg_type:
            print(f"  Step {step}: USER PROMPT")
            # Extract the user's question from the message
            print(f"          -> (user question sent to LLM)")
    
    elif "Response" in msg_type or "Model" in msg_type:
        step += 1
        msg_str = str(msg)
        
        # Check if this contains tool calls
        if "ToolCall" in msg_str or "tool_call" in msg_str or "tool-call" in msg_str:
            print(f"  Step {step}: LLM DECIDES to call tool")
            # Try to extract tool name
            if "query_sql" in msg_str or "run_sql" in msg_str:
                print(f"          -> Calling SQL tool")
        elif "ToolReturn" in msg_str or "tool_return" in msg_str or "tool-return" in msg_str:
            print(f"  Step {step}: TOOL RESULT returned")
            print(f"          -> {msg_str[:100]}")
        else:
            print(f"  Step {step}: LLM RESPONSE")
            print(f"          -> {msg_str[:100]}")

print(f"\nTotal steps: {step}")

Agent reasoning flow:

  Step 2: LLM DECIDES to call tool
          -> Calling SQL tool
  Step 4: LLM RESPONSE
          -> ModelResponse(parts=[ThinkingPart(content='\n\n', id='content', provider_name='openai'), TextPart(co

Total steps: 4


---

## Part 5: Cost Tracking in Detail

### How token estimation works

We don't have access to the exact token count from the API (PydanticAI
doesn't expose it directly). Instead we estimate:

```python
token_estimate = len(message_text) // 4  # ~4 chars per token for English
```

**How accurate is this?**

| Content Type | Chars/Token | Our Estimate Accuracy |
|-------------|-------------|----------------------|
| English text | ~4 | ~90% accurate |
| Code | ~3.5 | Slightly underestimates |
| JSON/structured | ~3 | Underestimates by ~15% |
| Non-English | ~2-3 | Can underestimate by 30%+ |

For production, use `tiktoken` (OpenAI) or the model's tokenizer for exact counts.
For learning and rough budgeting, the `len // 4` heuristic is good enough.

### Why track cost?

A single agent query can make 2-5 LLM calls (planning + tool calls + synthesis).
At $0.01-0.10 per query, 1000 queries/day = $10-100/day = $300-3000/month.
Cost tracking prevents surprises on your monthly bill.

In [11]:
# -----------------------------------------------------------------
# Token estimation and cost calculation
# -----------------------------------------------------------------

from analyst.observability.tracing import MODEL_PRICING

print("Model pricing (per 1M tokens):")
print(f"{'Model':<30} {'Input':>10} {'Output':>10}")
print("-" * 52)
for model, (inp, out) in MODEL_PRICING.items():
    print(f"{model:<30} ${inp:>8.2f} ${out:>8.2f}")

# Cost breakdown for our traced queries
print(f"\n\nCost breakdown for {len(deps.tracer.traces)} traced queries:")
total_cost = sum(t.total_cost_usd for t in deps.tracer.traces)
total_tokens = sum(t.total_tokens for t in deps.tracer.traces)

print(f"  Total tokens:  {total_tokens:,}")
print(f"  Total cost:    ${total_cost:.4f}")
print(f"  Cost per query: ${total_cost / len(deps.tracer.traces):.4f}")

# Project monthly cost
queries_per_day = 100
monthly_cost = total_cost / len(deps.tracer.traces) * queries_per_day * 30
print(f"\n  Projected monthly cost at {queries_per_day} queries/day: ${monthly_cost:.2f}")

Model pricing (per 1M tokens):
Model                               Input     Output
----------------------------------------------------
gpt-4o-mini                    $    0.15 $    0.60
gpt-4o                         $    2.50 $   10.00
claude-3-5-haiku-latest        $    0.80 $    4.00
claude-3-5-sonnet-latest       $    3.00 $   15.00


Cost breakdown for 5 traced queries:
  Total tokens:  0
  Total cost:    $0.0000
  Cost per query: $0.0000

  Projected monthly cost at 100 queries/day: $0.00


---

## Part 6: Error Analysis & Debugging

### Common agent failure modes

| Failure Mode | Symptom | Root Cause | Fix |
|-------------|---------|------------|-----|
| **Bad SQL** | `SQL error: no such column` | LLM hallucinated a column name | Add column list to system prompt |
| **Wrong tool** | Correct tool exists but not called | Unclear tool descriptions | Improve tool docstrings |
| **Infinite retry** | 3 retries, same error | Error message isn't actionable | Better error formatting in `ModelRetry` |
| **Timeout** | No answer after 30s | Complex query, slow model | Add timeout, use faster model |
| **Hallucination** | Confident but wrong numbers | LLM guessed instead of computing | Force tool use in system prompt |

### Debugging workflow

```
1. Find the failing trace
2. Look at the span timeline — which span failed?
3. If tool_call span failed → check the SQL/code
4. If llm_call span produced bad output → check the prompt
5. If no tool was called → LLM didn't understand it needs a tool
```

In [12]:
# -----------------------------------------------------------------
# Simulate some failures to see error analysis
# -----------------------------------------------------------------

# Query that will likely cause issues (ambiguous table reference)
tricky_questions = [
    "What is the average of column xyz?",  # Non-existent column
    "Join sales and employees on employee_id",  # No common key
]

for q in tricky_questions:
    print(f"\nQ: {q}")
    answer = traced_query(deps, q)
    print(f"A: {answer[:150]}")

# Error summary
errors = deps.tracer.get_error_summary()
if errors:
    print(f"\nError categories:")
    for err_type, count in errors.items():
        print(f"  {err_type}: {count}")

failures = deps.tracer.get_failures()
print(f"\nFailed traces: {len(failures)}/{len(deps.tracer.traces)}")
for f in failures:
    print(f"  - {f.question[:60]}")
    for s in f.errors:
        print(f"    Error in {s.name}: {s.error[:80]}")


Q: What is the average of column xyz?
A: 

The column `xyz` does not exist in the available tables (`sales` or `employees`). Please provide a valid column name from one of these tables.

Q: Join sales and employees on employee_id
A: 

The `sales` table does not contain an `employee_id` column based on the provided schema. Therefore, a direct join between `sales` and `employees` on

Error categories:
  Binder Error: 2

Failed traces: 0/7


---

## Part 7: Saving & Analyzing Traces

### Why persist traces?

1. **Post-mortem debugging** — User reports issue 2 hours later, you need the trace
2. **Trend analysis** — Is latency creeping up over time?
3. **Eval calibration** — Use real traces to build better eval cases
4. **Cost auditing** — Monthly cost reports per user/query type

### Storage options

| Option | Pros | Cons | Best For |
|--------|------|------|----------|
| **JSON files** | Simple, no dependencies | No querying, grows large | Prototyping |
| **SQLite** | Queryable, single file | Limited concurrency | Small deployments |
| **PostgreSQL** | Full SQL, concurrent | Setup overhead | Production |
| **Langfuse** | Purpose-built, dashboards | External dependency | Teams |

In [13]:
# -----------------------------------------------------------------
# Save traces for post-hoc analysis
# -----------------------------------------------------------------

traces_path = Path("../data/traces.json")
deps.tracer.save(traces_path)
print(f"Saved {len(deps.tracer.traces)} traces to {traces_path}")

# Load and analyze
saved_traces = json.loads(traces_path.read_text())
print(f"\nSample trace:")
print(json.dumps(saved_traces[0], indent=2)[:500])

Saved 7 traces to ../data/traces.json

Sample trace:
{
  "trace_id": "trace_0001",
  "question": "What is the total revenue in the sales data?",
  "model": "openai:local-model",
  "duration_ms": 70745.9,
  "total_tokens": 0,
  "total_cost_usd": 0.0,
  "success": true,
  "answer": "\n\nThe total revenue in the sales data is $147,695.10.",
  "spans": [
    {
      "name": "run_sql",
      "type": "tool_call",
      "duration_ms": 8.6,
      "tokens": 0,
      "error": null,
      "input": "SELECT SUM(revenue) AS total_revenue FROM sales",
      "out


In [14]:
# -----------------------------------------------------------------
# Analyze traces as a DataFrame (production pattern)
# -----------------------------------------------------------------

trace_data = []
for t in saved_traces:
    trace_data.append({
        "trace_id": t["trace_id"],
        "question": t["question"][:50],
        "duration_ms": t["duration_ms"],
        "tokens": t["total_tokens"],
        "cost_usd": t["total_cost_usd"],
        "success": t["success"],
        "n_spans": len(t["spans"]),
    })

traces_df = pd.DataFrame(trace_data)
print("Traces as DataFrame:\n")
print(traces_df.to_string(index=False))

print(f"\nSummary statistics:")
print(traces_df[["duration_ms", "tokens", "cost_usd"]].describe().round(2))

Traces as DataFrame:

  trace_id                                           question  duration_ms  tokens  cost_usd  success  n_spans
trace_0001       What is the total revenue in the sales data?      70745.9       0       0.0     True        1
trace_0002              Which region has the highest revenue?      45530.8       0       0.0     True        1
trace_0003          What is the average salary by department?      43681.5       0       0.0     True        1
trace_0004         How many products have revenue above 1000?      50942.6       0       0.0     True        1
trace_0005 What is the correlation between quantity and reven      45396.9       0       0.0     True        1
trace_0006                 What is the average of column xyz?      22234.2       0       0.0     True        0
trace_0007            Join sales and employees on employee_id      65511.7       0       0.0     True        2

Summary statistics:
       duration_ms  tokens  cost_usd
count         7.00     7.0      

---

## Part 8: Production Integration — Logfire & Langfuse

Our custom tracer is great for learning, but production systems use
dedicated observability platforms.

### Logfire (PydanticAI native)

Logfire is built by the Pydantic team and integrates with PydanticAI with
zero configuration:

```python
import logfire

logfire.configure()  # Uses LOGFIRE_TOKEN from .env
logfire.instrument_pydantic_ai()  # Auto-trace all PydanticAI agents

# That's it! Every agent.run_sync() is now traced in Logfire dashboard
agent = Agent("openai:local-model", system_prompt="...")
result = agent.run_sync("What is the total revenue?")  # Auto-traced
```

Logfire provides:
- Automatic span creation for LLM calls and tool calls
- Token counting and cost estimation
- Visual trace waterfall diagrams
- Performance dashboards

### Langfuse (Open-source)

Langfuse is an open-source LLM observability platform with broader
ecosystem support:

```python
from langfuse import Langfuse

langfuse = Langfuse()  # Uses LANGFUSE_* env vars

# Manual tracing with Langfuse
trace = langfuse.trace(name="data_analysis")
span = trace.span(name="llm_call", input={"question": "..."})
# ... run agent ...
span.end(output={"answer": "..."})
trace.update(output={"final_answer": "..."})
```

Langfuse provides:
- Trace visualization and search
- Eval dataset management
- A/B testing across model versions
- Cost analytics and alerting

### Which to Use? — Decision Guide

```
Are you using PydanticAI?
├── Yes → Start with Logfire (zero-config, built for PydanticAI)
│         └── Need advanced eval? → Add Langfuse alongside
└── No  → Use Langfuse (framework-agnostic, open-source)
          └── Need self-hosting? → Langfuse (self-hosted)
```

| Feature | Logfire | Langfuse |
|---------|---------|----------|
| PydanticAI integration | Zero-config (2 lines) | Manual instrumentation |
| Open source | No (hosted service) | Yes (self-host or cloud) |
| Eval features | Basic metrics | Full eval dataset management |
| Cost | Free tier available | Free self-hosted |
| Setup time | 2 minutes | 15-30 minutes |
| Best for | PydanticAI-native stacks | Multi-framework stacks |

### Our custom tracer vs production tools

| Feature | Our Tracer | Logfire/Langfuse |
|---------|-----------|-----------------|
| Learning value | High (you understand everything) | Low (magic black box) |
| Production-ready | No (in-memory, no UI) | Yes |
| Exact LLM timing | No (bundled) | Yes (per-call) |
| Dashboard | Text-based | Web UI |
| Team collaboration | No | Yes |

**Recommendation**: Use our tracer to learn, then switch to Logfire for
PydanticAI projects or Langfuse for multi-framework setups.

---

## Summary

| Concept | Implementation | Why |
|---------|---------------|-----|
| Spans | `Span` class with timing/metadata | Track individual operations |
| Traces | `Trace` groups spans per query | End-to-end view of agent run |
| AgentTracer | Collects traces, computes stats | Aggregate performance view |
| Message inspection | `result.all_messages()` | Debug exact LLM interactions |
| Cost tracking | Token count * model pricing | Budget control |
| Error analysis | `get_failures()`, `get_error_summary()` | Debug and improve |
| Persistence | Save traces to JSON | Post-hoc analysis |

### Production Patterns

1. **Trace everything** — You can't debug what you can't see
2. **Track latency percentiles** — P50 for typical, P95/P99 for worst case
3. **Set cost budgets** — Alert when per-query cost exceeds threshold
4. **Categorize errors** — Know which failure modes are most common
5. **Save traces** — You'll want them for debugging issues reported later
6. **Dashboard daily** — Catch regressions early (latency creep, cost spikes)

**Next: Lesson 8 — Full Agent Assembly (putting everything together)**

---

## Exercises

1. **Logfire integration** — Install logfire and instrument the traced agent. Compare the dashboard to our custom one.
2. **Alerting** — Add a function that raises a warning when P95 latency exceeds a threshold.
3. **Cost budget enforcement** — Modify the agent to stop mid-run if token cost exceeds a limit.
4. **Trace comparison** — Build a function that compares two sets of traces (before/after a change) and highlights regressions.
5. **Visual timeline** — Use matplotlib to create a Gantt-chart-style visualization of span timings.